# Lab 1
Create a free  Kaggle account if you havent already. 
Download the dataset https://www.kaggle.com/c/dogs-vs-cats/data
Place the images in folders: one for training cats, one for training dogs, and a smaller set for validation/testing

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import os
import shutil
import random
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt



## Part 1: Data Preparation
Organize the dataset into separate folders for training (80%), validation (10%), and test (10%).
Resize all images to the same size (for example, 150x150 or 224x224 pixels).
Apply data augmentation (like random flips or rotations) to make your model more robust.

In [47]:
random.seed(42)

base_dir = './data/PetImages'
output_dir = './data/split'
classes = ['Cat', 'Dog']
splits = {'train': 0.7, 'val': 0.15, 'test': 0.15}

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

for cls in classes:
    src_dir = os.path.join(base_dir, cls)
    images = [f for f in os.listdir(src_dir) if f.lower().endswith('.jpg')]
    random.shuffle(images)
    n_total = len(images)
    n_train = int(splits['train'] * n_total)
    n_val = int(splits['val'] * n_total)
    n_test = n_total - n_train - n_val

    split_files = {
        'train': images[:n_train],
        'val': images[n_train:n_train+n_val],
        'test': images[n_train+n_val:]
    }

    for split, files in split_files.items():
        for f in files:
            src = os.path.join(src_dir, f)
            dst = os.path.join(output_dir, split, cls, f)
            shutil.copy2(src, dst) 

print("Images have been split into train/val/test folders.")

Images have been split into train/val/test folders.


## Part 2: Building the CNN
Start by creating a simple CNN with a few convolution + pooling layers.
Add fully connected layers at the end for classification.
Use an activation function like ReLU in hidden layers and sigmoid/softmax in the output layer.

In [48]:

IMG_SIZE = (150, 150)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze the base

# Add your own classification head
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=IMG_SIZE + (3,)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/var/folders/n1/vds_7_gd3d1g9hj0w_5t2r4c0000gn/T/ipykernel_2922/2493203503.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_5 (Rescaling)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [49]:

def remove_invalid_images(folder):
    num_removed = 0
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith('.jpg'):
                fpath = os.path.join(root, fname)
                try:
                    img = Image.open(fpath)
                    img.verify()
                    img = Image.open(fpath)
                    if img.mode not in ('RGB', 'L'):
                        print(f"Removing non-RGB/non-grayscale image: {fpath} (mode: {img.mode})")
                        os.remove(fpath)
                        num_removed += 1
                except Exception:
                    print(f"Removing corrupted image: {fpath}")
                    os.remove(fpath)
                    num_removed += 1
    print(f"Removed {num_removed} invalid or corrupted images.")

remove_invalid_images('./data/split')

Removing non-RGB/non-grayscale image: ./data/split/test/Cat/660.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/test/Cat/10820.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/test/Dog/11410.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/test/Dog/1308.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/test/Dog/8730.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/test/Dog/7514.jpg (mode: RGBA)
Removing non-RGB/non-grayscale image: ./data/split/test/Dog/7459.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/train/Cat/11565.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/train/Cat/4833.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/train/Cat/2663.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/train/Cat/11210.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./data/split/train/Cat/12080.jpg (mode: P)
Removing non-RGB/non-grayscale image: ./

## Part 3: Training the Model
Compile the model with binary cross-entropy loss and an optimizer (like Adam).
Train the model for several epochs (start small, maybe 5–10).
Record the training and validation accuracy and loss after each epoch.

In [50]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
train_ds = tf.keras.utils.image_dataset_from_directory(
    './data/split/train',
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    './data/split/val',
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary'
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Found 17430 files belonging to 2 classes.
Found 3735 files belonging to 2 classes.
Epoch 1/10
  2/545 ━━━━━━━━━━━━━━━━━━━━ 3:37 400ms/step - accuracy: 0.5625 - loss: 0.7590

KeyboardInterrupt: 

## Part 4: Evaluation
Test the model on your unseen test set.
Report metrics: accuracy, precision, recall, and F1-score.
Create a confusion matrix to see where the model makes mistakes.

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    './data/split/test',
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary',
    shuffle=False
)

# Get true labels and predictions
y_true = np.concatenate([y for x, y in test_ds])
y_pred_probs = model.predict(test_ds)
y_pred = (y_pred_probs.flatten() > 0.5).astype(int)

# Metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Test Accuracy:  {acc:.3f}")
print(f"Precision:      {prec:.3f}")
print(f"Recall:         {rec:.3f}")
print(f"F1-score:       {f1:.3f}")
print("Confusion Matrix:")
print(cm)

Found 3745 files belonging to 2 classes.


NameError: name 'np' is not defined

## Part 5: Visualization
Pick a few test images and show the model’s predictions alongside the true labels.
Highlight at least 5 examples where the model was wrong and analyze why.

test_ds = tf.keras.utils.image_dataset_from_directory(
    './data/split/test',
    image_size=(150, 150),
    batch_size=32,
    label_mode='binary',
    shuffle=False
)

file_paths = test_ds.file_paths if hasattr(test_ds, 'file_paths') else None

images = []
labels = []
for batch_images, batch_labels in test_ds:
    images.append(batch_images.numpy())
    labels.append(batch_labels.numpy())
images = np.concatenate(images)
labels = np.concatenate(labels)

y_pred_probs = model.predict(test_ds)
y_pred = (y_pred_probs.flatten() > 0.5).astype(int)

wrong_idx = np.where(y_pred != labels)[0]

plt.figure(figsize=(15, 6))
for i, idx in enumerate(wrong_idx[:5]):
    plt.subplot(1, 5, i+1)
    plt.imshow(images[idx].astype("uint8"))
    plt.axis('off')
    plt.title(f"True: {int(labels[idx])}\nPred: {int(y_pred[idx])}")
plt.suptitle("5 Misclassified Test Images")
plt.show()

## Stretch Goals
Use transfer learning with a pre-trained network (like MobileNet or ResNet).
Compare the accuracy of your CNN vs. transfer learning.
Experiment with changing batch size, learning rate, or number of layers.